In [15]:
from gensim.models import Word2Vec, FastText
from tensorflow.keras import layers, Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

### Word2vec

In [2]:
sentences = [
    ['I', 'love', 'machine', 'learning'],
    ['Machine', 'learning', 'is', 'awesome'],
    ['Deep', 'learning', 'is', 'powerful'],
    ['I', 'enjoy', 'studying', 'machine', 'learning']
]

- Train Word2Vec

In [3]:
model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

- Get word vector

In [4]:
vector = model.wv['machine']

print(f"Vector shape: {vector.shape}")
print(f"Vector (first 10): {vector[:10]}")

Vector shape: (100,)
Vector (first 10): [ 9.4563962e-05  3.0773198e-03 -6.8126451e-03 -1.3754654e-03
  7.6685809e-03  7.3464094e-03 -3.6732971e-03  2.6427018e-03
 -8.3171297e-03  6.2054861e-03]


- Find similar words

In [5]:
similar = model.wv.most_similar('machine', topn=5)

for word, score in similar:
    print(f"{word}: {score:.3f}")

awesome: 0.199
studying: 0.170
powerful: 0.146
enjoy: 0.064
Deep: -0.003


- Word arithmetic (king - man + woman ≈ queen)

In [6]:
result = model.wv.most_similar(positive=['machine', 'learning'], negative=[], topn=3)

for word, score in result:
    print(f"  {word}: {score:.3f}")

  awesome: 0.164
  Machine: 0.133
  powerful: 0.118


- Save and load model

In [ ]:
model.save('word2vec.model')
loaded_model = Word2Vec.load('word2vec.model')

### Using Pre-trained Embeddings

**GloVe dictionary**

In [ ]:
def load_glove_embeddings(file_path):
    embeddings_index = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
            
    return embeddings_index

In [ ]:
embeddings_index = load_glove_embeddings('glove.6B.100d.txt')

print(f"Loaded {len(embeddings_index)} word vectors")

Loaded 400000 word vectors


In [11]:
text = "My name is Mohsin"

tokens = text.lower().split()
print(f"Tokens: {tokens}")

Tokens: ['my', 'name', 'is', 'mohsin']


In [12]:
word_vectors = []
embedding_dim = 100

for word in tokens:
    if word in embeddings_index:
        vector = embeddings_index[word]
    else:
        vector = np.zeros(embedding_dim)
        print(f"Word '{word}' not found in vocabulary. Using zero vector.")
        
    word_vectors.append(vector)


word_vectors = np.array(word_vectors)
print(f"\nShape of the text embedding: {word_vectors.shape}")


Shape of the text embedding: (4, 100)


2. Embedding matrix for Keras

In [22]:
def create_embedding_matrix(word_index, embeddings_index, embedding_dim=100):
    vocab_size = len(word_index) + 1
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    
    for word, i in word_index.items():
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
    
    return embedding_matrix, vocab_size

In [18]:
sentences = [
    "My name is Mohsin",
    "I am learning deep learning",
    "Word embeddings are fascinating"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

In [19]:
sequences = tokenizer.texts_to_sequences(sentences)

max_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

print(f"\nWord Index (Vocabulary): {tokenizer.word_index}")
print(f"Padded Sequences:\n{padded_sequences}\n")


Word Index (Vocabulary): {'learning': 1, 'my': 2, 'name': 3, 'is': 4, 'mohsin': 5, 'i': 6, 'am': 7, 'deep': 8, 'word': 9, 'embeddings': 10, 'are': 11, 'fascinating': 12}
Padded Sequences:
[[ 2  3  4  5  0]
 [ 6  7  1  8  1]
 [ 9 10 11 12  0]]



In [23]:
embedding_matrix, vocab_size = create_embedding_matrix(tokenizer.word_index, embeddings_index)

embedding_layer = layers.Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    input_length=max_length,
    trainable=False  # Freeze pre-trained embeddings
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
model = Sequential([
    embedding_layer,
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │         1,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,300 (5.08 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,300 (5.08 KB)

In [25]:
predictions = model.predict(padded_sequences)
print(f"\nModel Output Shape: {predictions.shape}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 369ms/step

Model Output Shape: (3, 1)


### FastText

- Train FastText

In [16]:
fasttext_model = FastText(sentences, vector_size=100, window=5, min_count=1)

unseen_vector = fasttext_model.wv['unseenword']

In [17]:
unseen_vector.shape

(100,)